# 99acres Property Scraper — Locality-Based

**Approach:** Scrapes **by locality** (sector/area) to bypass 99acres' 70-page city-wide limit. Auto-discovers all localities from the site's JSON facets, then scrapes each one.

**Supports:** Flats, Independent Houses, Independent Builder Floors

**Key features:**
- **Locality-based scraping** — auto-discovers all sectors/areas, scrapes each separately for full coverage (~30K+ listings vs ~1.5K city-wide)
- Fast — ~2 seconds per page, no Selenium/browser overhead
- Saves to CSV after every page — no data loss on IP block
- Auto-resumes (tracks completed localities + skips already-scraped property IDs)
- Filters by property type to avoid mixed results
- Uses `curl_cffi` to bypass Akamai bot detection

**Usage:** Change `PROPERTY_TYPE` in the config cell and run all cells.

In [8]:
from curl_cffi import requests as curl_requests
import pandas as pd
import json
import time
import random
import os

### Configuration

Set `PROPERTY_TYPE` to one of: `"flats"`, `"independent_house"`, `"builder_floor"`

In [ ]:

CITY = "gurgaon"

PROPERTY_TYPE = "flats"  

MAX_PAGES_PER_LOCALITY = 70  

# Throttling

MIN_DELAY = 1      # min seconds between page requests
MAX_DELAY = 3      # max seconds between page requests
BATCH_PAUSE = 20   # extra pause every BATCH_SIZE pages
BATCH_SIZE = 10


# URL prefix for each property type

URL_PREFIX_MAP = {
    "flats": "flats-in",
    "independent_house": "independent-house-in",
    "builder_floor": "independent-builder-floors-in",
}

FILE_PREFIX_MAP = {
    "flats": "flats",
    "independent_house": "independent_house",
    "builder_floor": "builder_floor",
}

PROPERTY_TYPE_FILTER = {
    "flats": "Residential Apartment",
    "independent_house": "Independent House/Villa",
    "builder_floor": "Builder Floor",
}

URL_PREFIX = URL_PREFIX_MAP[PROPERTY_TYPE]

CITY_BASE_URL = f"https://www.99acres.com/{URL_PREFIX}-{CITY}-ffid"

OUTPUT_DIR = "../../data/web_scraping"

OUTPUT_FILE = os.path.join(OUTPUT_DIR, f"{FILE_PREFIX_MAP[PROPERTY_TYPE]}_{CITY}.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Property Type: {PROPERTY_TYPE}")
print(f"Filter: PROPERTY_TYPE == '{PROPERTY_TYPE_FILTER[PROPERTY_TYPE]}'")
print(f"City: {CITY}")
print(f"City URL: {CITY_BASE_URL}")
print(f"Output: {OUTPUT_FILE}")

Property Type: flats
Filter: PROPERTY_TYPE == 'Residential Apartment'
City: gurgaon
City URL: https://www.99acres.com/flats-in-gurgaon-ffid
Output: ../../data/web_scraping\flats_gurgaon.csv


### Resume support + save helpers

Why do we need this? 99acres will rate-limit after ~50-100 requests. When that happens, we need to stop and re-run later. Without resume support, we'd lose everything and start from scratch.

load_scraped_ids() — Reads the existing CSV and loads all property IDs into a set. When we encounter a property we've already scraped, we skip it. This prevents duplicates across re-runs.

save_page_data() — Appends new records to CSV. Key detail: mode="a" means append, not overwrite. write_header is True only when the file doesn't exist yet (first write). This is why we save after EVERY page — if we get blocked at page 50, we still keep pages 1-49.

load_progress() / save_progress() — Tracks which localities (sectors) are fully scraped. Saves to a JSON file like .progress_flats_gurgaon.json. On re-run, we skip completed localities entirely — no wasted requests.

In [ ]:
def load_scraped_ids(output_file):
    if os.path.exists(output_file):
        df = pd.read_csv(output_file, on_bad_lines='skip')
        ids = set(df["property_id"].dropna().astype(str).tolist())
        print(f"Resuming: {len(ids)} properties already scraped.")
        return ids
    print("Fresh start.")
    return set()

def save_page_data(records, output_file):
    if not records:
        return
    df = pd.DataFrame(records)
    write_header = not os.path.exists(output_file)
    df.to_csv(output_file, mode="a", header=write_header, index=False)

PROGRESS_FILE = os.path.join(OUTPUT_DIR, f".progress_{FILE_PREFIX_MAP[PROPERTY_TYPE]}_{CITY}.json")

def load_progress():
    """Load set of completed locality slugs from progress file."""
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r") as f:
            progress = json.load(f)
        return set(progress.get("completed_localities", []))
    return set()

def save_progress(completed_localities):
    """Save completed locality slugs to progress file."""
    with open(PROGRESS_FILE, "w") as f:
        json.dump({"completed_localities": sorted(completed_localities)}, f)

scraped_ids = load_scraped_ids(OUTPUT_FILE)
completed_localities = load_progress()
print(f"Completed localities: {len(completed_localities)}")

Resuming: 28504 properties already scraped.
Completed localities: 101


### Mappings + property extractor

Why mappings? 99acres stores some values as numeric codes in their JSON:

Facing: "1" means East, "2" means West, etc.

Furnishing: "1" = Furnished, "2" = Semi-Furnished

Overlooking: "1" = Park/Garden, "2" = Main Road

We verified these by visiting actual property detail pages on the website and cross-referencing.

map_codes() — Some fields like OVERLOOKING have multiple values like "1,4" meaning "Park/Garden, Pool". 

This function splits them and maps each code.

extract_property(p) — The heart of data extraction. Takes one raw JSON property dict and converts it to our clean output format. Key transformations:

PROP_HEADING → property_name (e.g., "3 BHK Flat in Sector 85, Gurgaon")

PD_URL → link (prepend base URL)

SOCIETY_NAME or BUILDING_NAME → society (fallback logic)

FORMATTED_PRICE or PRICE → price (e.g., "2.5 Cr")

LOCALIZED_PRICE_SQFT_TEXT → area (e.g., "₹14,024 /sqft")

AREA + AREA_TYPE → areaWithType (e.g., "1640 sqft Superbuiltup Area")

FLOOR_NUM + TOTAL_FLOOR → floorNum (e.g., "6 of 26 Floors")

MAP_DETAILS.LATITUDE/LONGITUDE → lat/long (nested extraction)

USP_V2_FH_REMOVED → features (readable amenity names — NOT the numeric FEATURES field)

RESERVED_PARKING → parking (JSON like {"C":2,"O":1})

LANDMARK_DETAILS → nearbyLocations (list of nearby landmarks)

In [ ]:
FACING_MAP = {
    "1": "East", "2": "West", "3": "North", "4": "South",
    "5": "North-East", "6": "North-West", "7": "South-East", "8": "South-West",
}

FURNISH_MAP = {
    "1": "Furnished", "2": "Semi-Furnished", "3": "Unfurnished", "4": "Unfurnished",
}

OVERLOOKING_MAP = {
    "1": "Park/Garden", "2": "Main Road", "3": "Club",
    "4": "Pool", "5": "Others", "7": "Sea facing",
}

def map_codes(code_str, mapping):
    """Convert comma-separated numeric codes to readable names using a mapping dict."""
    if not code_str:
        return ""
    codes = str(code_str).split(",")
    names = [mapping.get(c.strip(), c.strip()) for c in codes]
    return ", ".join(names)

def extract_property(p):
    """Convert a raw JSON property dict into the output format."""
    landmarks = p.get("LANDMARK_DETAILS") or []
    nearby = [lm.get("name", "") for lm in landmarks if lm.get("name")]

    area_val = p.get("AREA", "")
    area_type = (p.get("AREA_TYPE") or "").replace("_", " ").title()
    area_with_type = f"{area_val} {area_type}".strip() if area_val else ""

    floor_num = p.get("FLOOR_NUM", "")
    total_floor = p.get("TOTAL_FLOOR", "")
    floor_info = f"{floor_num} of {total_floor} Floors" if floor_num and total_floor else str(floor_num)

    facing_code = str(p.get("FACING", ""))
    facing = FACING_MAP.get(facing_code, facing_code)

    furnish_code = str(p.get("FURNISH", ""))
    furnish = FURNISH_MAP.get(furnish_code, furnish_code)

    overlooking = map_codes(p.get("OVERLOOKING", ""), OVERLOOKING_MAP)

    features = p.get("USP_V2_FH_REMOVED", "")

    map_details = p.get("MAP_DETAILS") or {}
    latitude = map_details.get("LATITUDE", "")
    longitude = map_details.get("LONGITUDE", "")

    parking = p.get("RESERVED_PARKING", "")

    return {
        "property_name": p.get("PROP_HEADING", ""),
        "link": "https://www.99acres.com" + p.get("PD_URL", ""),
        "society": p.get("SOCIETY_NAME") or p.get("BUILDING_NAME", ""),
        "price": p.get("FORMATTED_PRICE") or p.get("PRICE", ""),
        "area": p.get("LOCALIZED_PRICE_SQFT_TEXT", ""),
        "areaWithType": area_with_type,
        "carpetArea": p.get("CARPET_AREA", ""),
        "bedRoom": p.get("BEDROOM_NUM", ""),
        "bathroom": p.get("BATHROOM_NUM", ""),
        "balcony": p.get("BALCONY_NUM", ""),
        "address": p.get("LOCALITY", ""),
        "floorNum": floor_info,
        "facing": facing,
        "overlooking": overlooking,
        "agePossession": p.get("AGE", ""),
        "cornerProperty": p.get("CORNER_PROPERTY", ""),
        "furnishing": furnish,
        "parking": parking,
        "nearbyLocations": nearby if nearby else "",
        "description": p.get("DESCRIPTION", ""),
        "features": features,
        "latitude": latitude,
        "longitude": longitude,
        "property_id": p.get("PROP_ID", ""),
    }

print("Extractor ready.")

Extractor ready.


### Discover localities

Fetches the city-wide listing page and extracts all locality/sector names from the JSON facets. Each locality becomes a separate scraping target.

In [ ]:
def discover_localities(city_url):
    """Fetch the city-wide listing page and extract locality facets."""
    print(f"Discovering localities from: {city_url}")
    resp = curl_requests.get(city_url, impersonate="chrome120", timeout=30)

    if resp.status_code != 200:
        raise Exception(f"Failed to fetch city page: HTTP {resp.status_code}")

    html = resp.text
    marker = "window.__initialData__="
    idx = html.find(marker)
    if idx == -1:
        raise Exception("No __initialData__ found on city page")

    decoder = json.JSONDecoder()
    data, _ = decoder.raw_decode(html[idx + len(marker):])
    localities = data["srp"]["pageData"]["facets"]["LOCALITY_ID"]

    results = []
    for loc in localities:
        slug = (loc["label"].lower()
                .strip()
                .replace(" ", "-")
                .replace("--", "-")
                .strip("-"))
        # Remove trailing city name to avoid duplication in URL
        if slug.endswith(f"-{CITY}"):
            slug = slug[:-len(f"-{CITY}")]
        results.append({
            "id": loc["id"],
            "label": loc["label"],
            "count": loc["count"],
            "slug": slug,
        })

    results.sort(key=lambda x: x["count"], reverse=True)
    return results

localities = discover_localities(CITY_BASE_URL)
total_listed = sum(loc["count"] for loc in localities)
pending = [loc for loc in localities if loc["slug"] not in completed_localities]

print(f"\nFound {len(localities)} localities with ~{total_listed} total listings")
print(f"Already completed: {len(completed_localities)}")
print(f"Remaining: {len(pending)}")
print(f"\nTop 10 localities by count:")
for loc in localities[:10]:
    status = "\u2705" if loc["slug"] in completed_localities else "\u23f3"
    print(f"  {status} {loc['label']}: {loc['count']} listings (slug: {loc['slug']})")

Discovering localities from: https://www.99acres.com/flats-in-gurgaon-ffid

Found 99 localities with ~29995 total listings
Already completed: 99
Remaining: 2

Top 10 localities by count:
  ✅ Sohna: 1147 listings (slug: sohna)
  ✅ Sector 102 Gurgaon: 1082 listings (slug: sector-102)
  ✅ Sector 79 Gurgaon: 1080 listings (slug: sector-79)
  ✅ Sector 106 Gurgaon: 974 listings (slug: sector-106)
  ✅ Sector 65 Gurgaon: 853 listings (slug: sector-65)
  ✅ Sector 104 Gurgaon: 827 listings (slug: sector-104)
  ✅ Sector 85 Gurgaon: 823 listings (slug: sector-85)
  ✅ Sector 92 Gurgaon: 816 listings (slug: sector-92)
  ✅ Sector 37D Gurgaon: 765 listings (slug: sector-37d)
  ✅ Sector 89 Gurgaon: 671 listings (slug: sector-89)


### Main scraper loop

Iterates through each locality, scraping all pages. Saves after every page so nothing is lost if the script gets blocked. Marks each locality as completed so re-runs skip finished ones.

In [12]:
MAX_CONSECUTIVE_ERRORS = 5
expected_type = PROPERTY_TYPE_FILTER[PROPERTY_TYPE]
total_new = 0
rate_limited = False
total_requests = 0

pending = [loc for loc in localities if loc["slug"] not in completed_localities]

for loc_idx, locality in enumerate(pending):
    slug = locality["slug"]
    label = locality["label"]
    base_url = f"https://www.99acres.com/{URL_PREFIX}-{slug}-{CITY}-ffid"

    print(f"\n{'='*60}")
    print(f"LOCALITY {loc_idx+1}/{len(pending)}: {label} (~{locality['count']} listings)")
    print(f"URL: {base_url}")
    print(f"{'='*60}")

    locality_new = 0
    consecutive_errors = 0

    for page in range(1, MAX_PAGES_PER_LOCALITY + 1):
        url = f"{base_url}-page-{page}" if page > 1 else base_url

        try:
            resp = curl_requests.get(url, impersonate="chrome120", timeout=30)
            total_requests += 1

            # HTTP error
            if resp.status_code != 200:
                if resp.status_code in (410, 404):
                    # 410/404 = no more pages for this locality
                    if page == 1:
                        print(f"  Page {page}: HTTP {resp.status_code} — URL may be invalid, skipping locality")
                    else:
                        print(f"  Page {page}: HTTP {resp.status_code} — end of pages")
                    break
                consecutive_errors += 1
                print(f"  Page {page}: HTTP {resp.status_code} (error {consecutive_errors}/{MAX_CONSECUTIVE_ERRORS})")
                if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                    print(f"  \u26d4 Rate limited! Stopping.")
                    rate_limited = True
                    break
                continue

            html = resp.text
            marker = "window.__initialData__="
            idx = html.find(marker)

            if idx == -1:
                consecutive_errors += 1
                print(f"  Page {page}: No JSON (error {consecutive_errors}/{MAX_CONSECUTIVE_ERRORS})")
                if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                    rate_limited = True
                    break
                continue

            decoder = json.JSONDecoder()
            data, _ = decoder.raw_decode(html[idx + len(marker):])

            if "srp" not in data:
                consecutive_errors += 1
                print(f"  Page {page}: No 'srp' key (error {consecutive_errors}/{MAX_CONSECUTIVE_ERRORS})")
                if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                    rate_limited = True
                    break
                continue

            properties = data["srp"]["pageData"]["properties"]
            individual = [
                p for p in properties
                if p.get("entityType") is None and p.get("PROPERTY_TYPE") == expected_type
            ]

            # No individual listings = end of useful pages
            if not individual and page > 1:
                print(f"  Page {page}: 0 listings — end of pages")
                break

            new_records = []
            for p in individual:
                pid = str(p.get("PROP_ID", ""))
                if pid and pid not in scraped_ids:
                    new_records.append(extract_property(p))
                    scraped_ids.add(pid)

            save_page_data(new_records, OUTPUT_FILE)
            locality_new += len(new_records)
            total_new += len(new_records)
            consecutive_errors = 0

            print(f"  Page {page}: {len(individual)} listings, {len(new_records)} new — locality: {locality_new}, total: {total_new}")

        except Exception as e:
            consecutive_errors += 1
            print(f"  Page {page}: Error — {e} (error {consecutive_errors}/{MAX_CONSECUTIVE_ERRORS})")
            if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                rate_limited = True
                break
            continue

        # Throttle
        if total_requests % BATCH_SIZE == 0:
            time.sleep(BATCH_PAUSE)
        else:
            time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

    # If not rate limited, mark locality as completed
    if not rate_limited:
        completed_localities.add(slug)
        save_progress(completed_localities)
        print(f"  \u2705 {label} done — {locality_new} new properties")
    else:
        print(f"\n\u26d4 Stopped due to rate limiting after {total_new} new properties.")
        print(f"\u2705 Progress saved ({len(completed_localities)} localities done). Just re-run to resume.")
        break

    # Extra pause between localities
    if loc_idx < len(pending) - 1:
        pause = random.uniform(5, 10)
        print(f"  Switching locality (pause {pause:.0f}s)...")
        time.sleep(pause)

print(f"\n{'='*60}")
print(f"SUMMARY: {total_new} new properties scraped across {len(completed_localities)} localities")
print(f"Total unique properties: {len(scraped_ids)}")
print(f"Output: {OUTPUT_FILE}")
print(f"{'='*60}")


SUMMARY: 0 new properties scraped across 101 localities
Total unique properties: 28536
Output: ../../data/web_scraping\flats_gurgaon.csv


### Verify & deduplicate

Load the CSV, drop any duplicate property IDs, and show a summary.

In [13]:
df = pd.read_csv(OUTPUT_FILE, on_bad_lines='warn')
print(f"Total rows: {len(df)}")

before = len(df)
df = df.drop_duplicates(subset="property_id", keep="first")
after = len(df)

if before != after:
    print(f"Removed {before - after} duplicates.")
    df.to_csv(OUTPUT_FILE, index=False)

print(f"Final: {len(df)} unique properties")
print(f"\nColumns ({len(df.columns)}): {list(df.columns)}")
print(f"\nNull counts:")
print(df.isnull().sum())
df.head()


ParserError: Error tokenizing data. C error: Expected 24 fields in line 28506, saw 25
